# 🧬 pg_bio: Ultra-Fast Native Sequence Alignment
In this tutorial, we demonstrate how to use `rust-bio` powered Smith-Waterman local sequence alignments directly in Postgres.

In [ ]:
import pandas as pd
from pgbio import PgBioClient
import psycopg

# Connect to pg_bio
conn_str = "postgresql://localhost:28818/pg_bio"
client = PgBioClient(conn_str)

# Seed the database with some example proteins
with client._get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE TABLE IF NOT EXISTS proteins (uniprot_id TEXT, name TEXT, sequence TEXT, embedding halfvec);")
        cur.execute("TRUNCATE proteins;")
        cur.execute("INSERT INTO proteins (uniprot_id, name, sequence) VALUES ('P12345', 'Protein A', 'MKTLLALALALALALALALALALALALALALALALA');")
        cur.execute("INSERT INTO proteins (uniprot_id, name, sequence) VALUES ('P67890', 'Protein B', 'MKTLLXYZXYZXYZXYZXYZXYZXYZXYZXYZXYZX');")
        cur.execute("INSERT INTO proteins (uniprot_id, name, sequence) VALUES ('Q11111', 'Protein C', 'MMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMM');")
        conn.commit()


## Native SDK Alignment
Now, we run the native sequence alignment function. It runs entirely inside the database using Rust!

In [ ]:
# Search for the MKTLL motif with high local alignment scores
matches = client.align_sequence("MKTLL", threshold=10, limit=10)
df = pd.DataFrame(matches)
df
